# Example 0.9.2 / 1: Solve an Instance

Companion notebook for **0.9.2 The Transportation Model** in the AMPL lab Documentation, section "Solve an Instance". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Solve an Instance

The AMPL book's steel example ships from three mills to seven factories; total supply equals total demand, 6900 tons. In `data.py` and `run.py`, `ampl` is an `AMPL` object that has already read `model.mod`. The cost table is a pandas DataFrame with origins as rows; `stack()` turns it into the `(origin, destination)` keys the two-dimensional `cost` needs:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ORIG;                          # Shipping origins
set DEST;                          # Delivery destinations
param supply {ORIG} >= 0;          # Units available
param demand {DEST} >= 0;          # Units required
param cost {ORIG, DEST} >= 0;      # Cost per shipped unit
var Ship {ORIG, DEST} >= 0;        # Units on each route
minimize TransportCost:
    sum {i in ORIG, j in DEST} cost[i,j] * Ship[i,j];
subject to SupplyLimit {i in ORIG}:
    sum {j in DEST} Ship[i,j] <= supply[i];
subject to DemandRequirement {j in DEST}:
    sum {i in ORIG} Ship[i,j] >= demand[j];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
import pandas as pd

ampl.set["ORIG"] = ["GARY", "CLEV", "PITT"]
ampl.set["DEST"] = ["FRA", "DET", "LAN", "WIN", "STL", "FRE", "LAF"]
ampl.param["supply"] = {"GARY": 1400, "CLEV": 2600, "PITT": 2900}
ampl.param["demand"] = {"FRA": 900, "DET": 1200, "LAN": 600, "WIN": 400,
                        "STL": 1700, "FRE": 1100, "LAF": 1000}
cost = pd.DataFrame(
    [[39, 14, 11, 14, 16, 82, 8],
     [27, 9, 12, 9, 26, 95, 17],
     [24, 14, 17, 13, 28, 99, 20]],
    index=["GARY", "CLEV", "PITT"],
    columns=["FRA", "DET", "LAN", "WIN", "STL", "FRE", "LAF"],
)
ampl.param["cost"] = cost.stack().to_dict()

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print(ampl.obj["TransportCost"].value())
ship = ampl.var["Ship"].to_pandas()
print(ship[ship["Ship.val"] > 0])   # only the routes used

HiGHS 1.14.0: optimal solution; objective 196200
13 simplex iterations
0 barrier iterations
solved
196200.0
               Ship.val
index0 index1          
CLEV   DET         1200
       LAF          400
       LAN          600
       WIN          400
GARY   FRE         1100
       LAF          300
PITT   FRA          900
       LAF          300
       STL         1700


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`. In the data file, `cost` is a table with the origins as rows:

In [6]:
%%writefile data.dat
param: ORIG: supply :=
  GARY 1400   CLEV 2600   PITT 2900 ;
param: DEST: demand :=
  FRA 900   DET 1200   LAN 600   WIN 400
  STL 1700  FRE 1100   LAF 1000 ;
param cost:  FRA  DET  LAN  WIN  STL  FRE  LAF :=
  GARY        39   14   11   14   16   82    8
  CLEV        27    9   12    9   26   95   17
  PITT        24   14   17   13   28   99   20 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display solve_result, TransportCost;
display {i in ORIG, j in DEST: Ship[i,j] > 0} Ship[i,j];

HiGHS 1.14.0: optimal solution; objective 196200
13 simplex iterations
0 barrier iterations
solve_result = solved
TransportCost = 196200

Ship[i,j] :=
CLEV DET   1200
CLEV LAF    400
CLEV LAN    600
CLEV WIN    400
GARY FRE   1100
GARY LAF    300
PITT FRA    900
PITT LAF    300
PITT STL   1700
;

